In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

In [3]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)

In [4]:
df=pd.read_csv(r"C:\Users\acann\OneDrive\Desktop\mtech major project\Anwesha Chakraborty\output\Final_Dataset.csv")

In [5]:
df["Datetime"] = pd.to_datetime(df["Datetime"])

In [6]:
print(df.shape)
print(df.head())
print(df.dtypes)

(21687, 7)
             Datetime  PM2.5_Mean  Visibility     RH  Temperature  WindSpeed  \
0 2024-01-01 00:00:00  170.053667    1207.008  87.56         12.0   2.056384   
1 2024-01-01 01:00:00  169.284333    1207.008  87.47         11.0   0.000000   
2 2024-01-01 02:00:00  165.553000    1207.008  87.47         11.0   2.056384   
3 2024-01-01 03:00:00  163.278667    1207.008  87.47         11.0   1.542288   
4 2024-01-01 04:00:00  155.996000    1207.008  87.47         11.0   1.542288   

          BLH  
0   39.382355  
1   47.932030  
2   57.352417  
3   62.579464  
4  135.569460  
Datetime       datetime64[ns]
PM2.5_Mean            float64
Visibility            float64
RH                    float64
Temperature           float64
WindSpeed             float64
BLH                   float64
dtype: object


In [7]:
print(df.isnull().sum())

Datetime       0
PM2.5_Mean     0
Visibility     0
RH             0
Temperature    0
WindSpeed      0
BLH            0
dtype: int64


In [8]:
features = [
    "PM2.5_Mean",
    "RH",
    "Temperature",
    "WindSpeed",
    "BLH"
]

target = "Visibility"

In [9]:
df["Visibility_1h"] = df[target].shift(-1)
df["Visibility_3h"] = df[target].shift(-3)
df["Visibility_6h"] = df[target].shift(-6)

In [11]:
df["Target_Time_1h"] = df["Datetime"].shift(-1)
df["Target_Time_3h"] = df["Datetime"].shift(-3)
df["Target_Time_6h"] = df["Datetime"].shift(-6)

In [12]:
df_forecast = df.dropna(
    subset=[
        "Visibility_1h",
        "Visibility_3h",
        "Visibility_6h"
    ]
).copy()
print(df_forecast.shape)

(21681, 13)


In [14]:
fog_months = [1, 2, 3, 11, 12]

case1_train_mask = (
    (df_forecast["Datetime"].dt.year.isin([2024, 2025])) &
    (df_forecast["Datetime"].dt.month.isin(fog_months)))

In [16]:
case2_train_mask = (
    df_forecast["Datetime"].dt.year.isin([2024, 2025]))

In [17]:
def get_test_mask(df, lead_time):

    target_time_col = f"Target_Time_{lead_time}h"

    test_mask = (
        (df[target_time_col] >= "2026-01-01") &
        (df[target_time_col] < "2026-04-01")
    )

    return test_mask

In [18]:
def get_models():

    models = {

        "Linear Regression": {
            "model": LinearRegression(),
            "scale": True
        },

        "Ridge Regression": {
            "model": Ridge(alpha=1),
            "scale": True
        },

        "Random Forest": {
            "model": RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            ),
            "scale": False
        },

        "Gradient Boosting": {
            "model": GradientBoostingRegressor(
                n_estimators=300,
                random_state=42
            ),
            "scale": False
        },

        "SVR": {
            "model": SVR(
                kernel="rbf",
                C=100,
                epsilon=0.1
            ),
            "scale": True
        }
    }

    return models

In [19]:
def run_forecast(
    df,
    model,
    scale_features,
    features,
    lead_time,
    train_mask,
    test_mask
):

    target_column = f"Visibility_{lead_time}h"

    X_train = df.loc[
        train_mask,
        features
    ].copy()

    y_train = df.loc[
        train_mask,
        target_column
    ].copy()

    X_test = df.loc[
        test_mask,
        features
    ].copy()

    y_test = df.loc[
        test_mask,
        target_column
    ].copy()


    train_data = pd.concat(
        [X_train, y_train],
        axis=1
    ).dropna()

    test_data = pd.concat(
        [X_test, y_test],
        axis=1
    ).dropna()


    X_train = train_data[features]
    y_train = train_data[target_column]

    X_test = test_data[features]
    y_test = test_data[target_column]


    if scale_features:

        scaler = StandardScaler()

        X_train_model = scaler.fit_transform(
            X_train
        )

        X_test_model = scaler.transform(
            X_test
        )

    else:

        X_train_model = X_train
        X_test_model = X_test

    model.fit(
        X_train_model,
        y_train
    )

    y_pred = model.predict(
        X_test_model
    )

    mae = mean_absolute_error(
        y_test,
        y_pred
    )

    mse = mean_squared_error(
        y_test,
        y_pred
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        y_test,
        y_pred
    )
    return {
        "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2, "Actual": y_test, "Predicted": y_pred
    }

In [20]:
lead_times = [1, 3, 6]

In [21]:
training_cases = {
    "Case 1 - Fog Season": case1_train_mask,
    "Case 2 - All Months": case2_train_mask
}


In [22]:
models = get_models()
all_forecast_results = []
all_forecast_predictions = {}

In [23]:
for case_name, train_mask in training_cases.items():

    for lead_time in lead_times:

        # Test mask for this lead time
        test_mask = get_test_mask(
            df_forecast,
            lead_time
        )
        for model_name, model_info in models.items():
            print(
                f"Running: "
                f"{model_name} | "
                f"{case_name} | "
                f"{lead_time}-hour forecast"
            )
            result = run_forecast(
                df=df_forecast,
                model=model_info["model"],
                scale_features=model_info["scale"],
                features=features,
                lead_time=lead_time,
                train_mask=train_mask,
                test_mask=test_mask
            )
            # Store numerical results
            all_forecast_results.append({
                "Model": model_name,
                "Training Case": case_name,
                "Lead Time (hours)": lead_time,
                "MAE": result["MAE"],
                "MSE": result["MSE"],
                "RMSE": result["RMSE"],
                "R2": result["R2"]
            })
            # Store predictions
            prediction_key = (
                f"{model_name}_"
                f"{case_name}_"
                f"{lead_time}h"
            )
            all_forecast_predictions[
                prediction_key
            ] = {
                "Actual": result["Actual"],
                "Predicted": result["Predicted"]
            }

Running: Linear Regression | Case 1 - Fog Season | 1-hour forecast
Running: Ridge Regression | Case 1 - Fog Season | 1-hour forecast
Running: Random Forest | Case 1 - Fog Season | 1-hour forecast
Running: Gradient Boosting | Case 1 - Fog Season | 1-hour forecast
Running: SVR | Case 1 - Fog Season | 1-hour forecast
Running: Linear Regression | Case 1 - Fog Season | 3-hour forecast
Running: Ridge Regression | Case 1 - Fog Season | 3-hour forecast
Running: Random Forest | Case 1 - Fog Season | 3-hour forecast
Running: Gradient Boosting | Case 1 - Fog Season | 3-hour forecast
Running: SVR | Case 1 - Fog Season | 3-hour forecast
Running: Linear Regression | Case 1 - Fog Season | 6-hour forecast
Running: Ridge Regression | Case 1 - Fog Season | 6-hour forecast
Running: Random Forest | Case 1 - Fog Season | 6-hour forecast
Running: Gradient Boosting | Case 1 - Fog Season | 6-hour forecast
Running: SVR | Case 1 - Fog Season | 6-hour forecast
Running: Linear Regression | Case 2 - All Months | 1

In [24]:
 forecast_results_df = pd.DataFrame(
    all_forecast_results
)
forecast_results_df = forecast_results_df.round(3)
forecast_results_df

,Model,Training Case,Lead Time (hours),MAE,MSE,RMSE,R2
0,Linear Regression,Case 1 - Fog Season,1,782.834,932468.984,965.644,0.603
1,Ridge Regression,Case 1 - Fog Season,1,782.824,932439.202,965.629,0.603
2,Random Forest,Case 1 - Fog Season,1,712.666,808222.833,899.012,0.656
3,Gradient Boosting,Case 1 - Fog Season,1,706.915,805489.924,897.491,0.657
4,SVR,Case 1 - Fog Season,1,695.020,763677.521,873.886,0.675
5,Linear Regression,Case 1 - Fog Season,3,787.923,1002990.626,1001.494,0.573
6,Ridge Regression,Case 1 - Fog Season,3,787.905,1002946.892,1001.472,0.573
7,Random Forest,Case 1 - Fog Season,3,746.178,906596.922,952.154,0.614
8,Gradient Boosting,Case 1 - Fog Season,3,743.134,898732.633,948.015,0.617
9,SVR,Case 1 - Fog Season,3,721.063,859874.258,927.294,0.634


In [28]:
comparison_table = forecast_results_df.pivot_table(
    index=["Model", "Training Case"],
    columns="Lead Time (hours)",
    values=["MAE", "RMSE", "R2"]
)

comparison_table = comparison_table.swaplevel(0, 1, axis=1)
comparison_table = comparison_table.reindex(
    columns=pd.MultiIndex.from_product(
        [[1, 3, 6], ["MAE", "RMSE", "R2"]]
    )
)

comparison_table.round(3)

1                        3  \
                                           MAE     RMSE     R2      MAE   
Model             Training Case                                           
Gradient Boosting Case 1 - Fog Season  706.915  897.491  0.657  743.134   
                  Case 2 - All Months  707.948  896.206  0.658  722.919   
Linear Regression Case 1 - Fog Season  782.834  965.644  0.603  787.923   
                  Case 2 - All Months  796.605  974.773  0.596  797.088   
Random Forest     Case 1 - Fog Season  712.666  899.012  0.656  746.178   
                  Case 2 - All Months  720.722  908.937  0.648  739.934   
Ridge Regression  Case 1 - Fog Season  782.824  965.629  0.603  787.905   
                  Case 2 - All Months  796.614  974.783  0.596  797.095   
SVR               Case 1 - Fog Season  695.020  873.886  0.675  721.063   
                  Case 2 - All Months  685.865  873.246  0.675  712.574   

                                                              6            \
                                           RMSE     R2      MAE      RMSE   
Model             Training Case                                             
Gradient Boosting Case 1 - Fog Season   948.015  0.617  944.539  1231.141   
                  Case 2 - All Months   925.653  0.635  885.862  1179.616   
Linear Regression Case 1 - Fog Season  1001.494  0.573  971.114  1289.426   
                  Case 2 - All Months  1009.749  0.566  953.469  1245.788   
Random Forest     Case 1 - Fog Season   952.154  0.614  950.576  1241.256   
                  Case 2 - All Months   944.894  0.620  920.364  1222.162   
Ridge Regression  Case 1 - Fog Season  1001.472  0.573  971.101  1289.405   
                  Case 2 - All Months  1009.756  0.566  953.472  1245.788   
SVR               Case 1 - Fog Season   927.294  0.634  955.348  1292.619   
                  Case 2 - All Months   933.263  0.629  909.959  1244.364   

                                              
                                          R2  
Model             Training Case               
Gradient Boosting Case 1 - Fog Season  0.355  
                  Case 2 - All Months  0.408  
Linear Regression Case 1 - Fog Season  0.292  
                  Case 2 - All Months  0.339  
Random Forest     Case 1 - Fog Season  0.344  
                  Case 2 - All Months  0.364  
Ridge Regression  Case 1 - Fog Season  0.292  
                  Case 2 - All Months  0.339  
SVR               Case 1 - Fog Season  0.289  
                  Case 2 - All Months  0.341

In [29]:
best_models = (
    forecast_results_df
    .loc[
        forecast_results_df
        .groupby(
            [
                "Training Case",
                "Lead Time (hours)"
            ]
        )["MAE"]
        .idxmin()
    ]
    .sort_values(
        [
            "Training Case",
            "Lead Time (hours)"
        ]
    )
)
best_models

,Model,Training Case,Lead Time (hours),MAE,MSE,RMSE,R2
4,SVR,Case 1 - Fog Season,1,695.020,763677.521,873.886,0.675
9,SVR,Case 1 - Fog Season,3,721.063,859874.258,927.294,0.634
13,Gradient Boosting,Case 1 - Fog Season,6,944.539,1515708.512,1231.141,0.355
19,SVR,Case 2 - All Months,1,685.865,762558.560,873.246,0.675
24,SVR,Case 2 - All Months,3,712.574,870979.871,933.263,0.629
28,Gradient Boosting,Case 2 - All Months,6,885.862,1391494.797,1179.616,0.408
